# Week 22 · Notebook 1: PySpark Data Engineering

# Requirements: Databricks workspace (free trial) + upload the week-01 CSVs to a volume

Upload into `/Volumes/zrl_/zorologistics/raw/`:

- `shipments.csv`
- `carriers.csv`
- `lanes.csv`

Then run on an all-purpose cluster. This notebook expresses the same cleaning/KPI logic twice, PySpark DataFrames and Spark SQL, and asserts the two agree.


## The PySpark mental model

A **DataFrame** is a named-column, immutable table. The single most important idea is **lazy evaluation**: `select`, `filter`, `join`, `groupBy`, `withColumn` are *transformations* that only build a logical plan; `count`, `show`, `collect`, `saveAsTable` are *actions* that run it. In production the write is usually the only action, because each extra action interrupts the optimizer. See `reference/platforms/databricks/05-pyspark.md`.


In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

VOL = "/Volumes/zrl_/zorologistics/raw"

ship = (spark.read.format("csv").option("header", True).option("inferSchema", True)
        .load(f"{VOL}/shipments.csv"))
car  = (spark.read.format("csv").option("header", True).option("inferSchema", True)
        .load(f"{VOL}/carriers.csv"))
lan  = (spark.read.format("csv").option("header", True).option("inferSchema", True)
        .load(f"{VOL}/lanes.csv"))

print("shipments:", ship.count(), "rows")
print("carriers: ", car.count(), "rows")
print("lanes:    ", lan.count(), "rows")
ship.printSchema()


## Transformations: cast, derive, dedupe, impute

Week 2's pandas cleaning, re-expressed in Spark. Timestamps are parsed with an optional-fraction format because the generator can emit sub-second values. `is_on_time` is recomputed from `delay_hours` (the CSV's `True`/`False` strings are avoided). `dropDuplicates` dedupes; `fillna` imputes.


In [ ]:
ship = (ship
  .withColumn("planned_departure", F.to_timestamp("planned_departure", "yyyy-MM-dd HH:mm:ss[.SSSSSS]"))
  .withColumn("planned_arrival",   F.to_timestamp("planned_arrival",   "yyyy-MM-dd HH:mm:ss[.SSSSSS]"))
  .withColumn("actual_arrival",    F.to_timestamp("actual_arrival",    "yyyy-MM-dd HH:mm:ss[.SSSSSS]"))
  .withColumn("weight_kg",  F.col("weight_kg").cast("double"))
  .withColumn("value_usd",  F.col("value_usd").cast("double"))
  .withColumn("delay_hours", F.col("delay_hours").cast("double"))
  .withColumn("is_on_time", F.col("delay_hours") <= 2.0)
  .withColumn("transit_hours",
      (F.unix_timestamp("actual_arrival") - F.unix_timestamp("planned_departure")) / 3600.0)
)

# Dedupe on shipment_id, then impute the planted NaN weights (mean = 850).
ship = ship.dropDuplicates(["shipment_id"])
ship = ship.fillna({"weight_kg": 850.0})

ship.select("shipment_id", "carrier_id", "lane_id", "delay_hours", "is_on_time", "transit_hours") \
    .show(5, truncate=False)


## Joins

Bring in carrier and lane dimensions with a left join. `carriers.on_time_rate` is the ground-truth reliability; `lanes.distance_km` still has the planted `NaN`s, which become `NULL`.


In [ ]:
enriched = (ship
  .join(car.select("carrier_id", "carrier_name", "region", "on_time_rate"),
        "carrier_id", "left")
  .join(lan.select("lane_id", "origin", "destination", "distance_km"),
        "lane_id", "left")
)
enriched.select("shipment_id", "carrier_name", "origin", "destination", "distance_km") \
    .show(5, truncate=False)


## Aggregations

On-time KPIs by carrier/lane/month, the same shape as Week 21's gold, built with the DataFrame API. `F.when(...).otherwise(...)` turns a boolean into 0/1 before averaging.


In [ ]:
kpis = (enriched
  .withColumn("month", F.date_format("actual_arrival", "yyyy-MM"))
  .groupBy("carrier_id", "carrier_name", "lane_id", "origin", "destination", "month")
  .agg(
      F.count("*").alias("shipment_count"),
      F.round(F.avg(F.when(F.col("is_on_time"), 1.0).otherwise(0.0)), 4).alias("on_time_rate"),
      F.round(F.avg("delay_hours"), 2).alias("avg_delay_hours"),
      F.round(F.sum("value_usd"), 2).alias("total_value_usd"),
  )
)
kpis.orderBy("on_time_rate").show(5, truncate=False)


## Window functions

A window computes across related rows *without collapsing them*. Here we rank carriers by on-time rate *within each month*, "who is winning this month".


In [ ]:
w = Window.partitionBy("month").orderBy(F.desc("on_time_rate"))
kpis = kpis.withColumn("rank_in_month", F.row_number().over(w))
kpis.filter(F.col("rank_in_month") <= 2).orderBy("month", "rank_in_month") \
    .select("month", "carrier_name", "lane_id", "on_time_rate", "rank_in_month") \
    .show(10, truncate=False)


## Write Delta, partitioned by month

The write is the *action* that materializes everything. Partitioning by `month` makes month-range filters cheap, though on Databricks, liquid clustering is the modern alternative to Hive partitioning.


In [ ]:
(kpis.write
  .mode("overwrite")
  .partitionBy("month")
  .saveAsTable("zrl_.zorologistics.gold_on_time_kpis_spark"))

print("gold rows:", spark.table("zrl_.zorologistics.gold_on_time_kpis_spark").count())


## The same logic in SQL, for comparison

Register the enriched DataFrame as a temporary view and write the identical aggregation in Spark SQL. The two approaches must agree to floating-point tolerance, that agreement is your cross-check.


In [ ]:
enriched.createOrReplaceTempView("enriched")
sql_kpis = spark.sql("""
SELECT carrier_id, carrier_name, lane_id, origin, destination,
       date_format(actual_arrival, 'yyyy-MM') AS month,
       count(*) AS shipment_count,
       round(avg(CASE WHEN is_on_time THEN 1.0 ELSE 0.0 END), 4) AS on_time_rate,
       round(avg(delay_hours), 2) AS avg_delay_hours
FROM enriched
GROUP BY 1, 2, 3, 4, 5, 6
""")
sql_kpis.orderBy("on_time_rate").show(5, truncate=False)


In [ ]:
%sql
-- The identical aggregation, written as a %sql cell against the same temp view.
-- Compare this output with the PySpark kpis DataFrame above, the numbers must match.
SELECT carrier_id, carrier_name, lane_id, origin, destination,
       date_format(actual_arrival, 'yyyy-MM') AS month,
       count(*) AS shipment_count,
       round(avg(CASE WHEN is_on_time THEN 1.0 ELSE 0.0 END), 4) AS on_time_rate,
       round(avg(delay_hours), 2) AS avg_delay_hours
FROM enriched
GROUP BY 1, 2, 3, 4, 5, 6
ORDER BY on_time_rate ASC
LIMIT 5


In [ ]:
# Cross-check: average on-time rate from both paths must match.
py_avg  = kpis.agg(F.avg("on_time_rate")).collect()[0][0]
sql_avg = sql_kpis.agg(F.avg("on_time_rate")).collect()[0][0]
print("PySpark avg on-time rate:", round(py_avg, 4))
print("SQL     avg on-time rate:", round(sql_avg, 4))
print("match:", abs(py_avg - sql_avg) < 1e-6)


In [ ]:
# Final metric: overall on-time rate and enriched row count.
final_rate = enriched.agg(F.avg(F.when(F.col("is_on_time"), 1.0).otherwise(0.0))).collect()[0][0]
print("final on-time rate:", round(final_rate, 4))
print("enriched rows:", enriched.count())
